# 원판 입력 크기 640·800 비교
같은 베이스라인 `best.pt`로 사람별 검증 영상의 첫 프레임부터 추론하고, 무릎 통과~접촉 구간을 비교합니다. 선명도는 트레이너가 보이는 바깥 테두리로 판독한 값입니다. 고정 박스 검증셋의 mAP도 별도로 확인합니다.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT
from wlcv.evaluation.plate_resolution import (
    run_resolution, fixed_validation, paired_frames, summarize, plot_pair, fiftyone_dataset,
)
# 같은 실행 이름은 저장된 결과를 다시 연다. 새로 측정하려면 이름을 바꾼다.
run_name = "2026-11-24"
base = ROOT / "outputs/plate_resolution"
output = base / run_name
if not output.exists():
    base.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=base) as temporary:
        stage = Path(temporary) / "run"
        run_resolution(stage)
        fixed_validation(stage)
        stage.rename(output)
detail = pd.read_csv(output / "frame_results.csv")
times = pd.read_csv(output / "timings.csv")
pair = paired_frames(detail)
print(f"시간 측정 {len(times)//2}프레임 × 2종 · 구간 판독 {len(pair)//2}장")

## 정탐과 허브 위치
`YOLO`는 신뢰도 0.10 이상 직접 검출이며 `Past link`는 고신뢰 검출 뒤 최대 2프레임 안에서 직전 박스 폭의 0.5배 반경에 든 저신뢰 후보까지 연결합니다. 검수 박스 IoU 0.50 이상을 정탐으로 셉니다. `abs_y`는 두 입력이 **모두 정탐**이고 직접 찍은 허브가 있는 프레임에서만 계산한 절대 세로 오차의 중앙값(px)입니다. `복구·손실`은 640에서 800으로 바꾼 결과입니다.

In [ ]:
pair.to_csv(output / "paired_frames.csv", index=False)
display(summarize(pair))
for group in ["method", "clarity", "person_id", "occlusion"]:
    groups = [group] if group == "method" else ["method", group]
    table = summarize(pair, groups)
    name = {"method": "summary", "person_id": "person"}.get(group, group)
    table.to_csv(output / f"{name}.csv", index=False)
    if group != "method":
        display(table[table.method == "Past link"])
linked = pair[pair.method == "Past link"]
columns = ["video_id", "frame_index", "clarity", "change", "source_640", "source_800", "iou_640", "iou_800"]
display(linked.loc[linked.change.isin(["복구", "손실"]), columns])

## Mac 지연 시간과 고정 검증셋
시간 단위는 ms/프레임입니다. CPU·배치 1, 영상마다 크기별 3회 예열 후 매 프레임 실행 순서를 번갈아 측정합니다. 시간에는 RGB 입력 변환부터 예측 반환까지 포함됩니다. 영상 읽기와 시간 연결은 제외합니다. mAP는 기존 라벨이 있는 전체 검증셋에서 별도로 계산합니다.

In [ ]:
display(times.groupby("imgsz").ms.quantile([.5, .95]).unstack().rename(columns={.5: "중앙값(ms)", .95: "95백분위(ms)"}))
display(pd.read_csv(output / "fixed_validation.csv"))

## 같은 장면을 나란히 보기
영상 ID·프레임 번호를 바꾸면 다른 선수도 확인할 수 있습니다. 노란 점선은 수동 박스, 노란 원은 수동 허브, 파란 박스와 ×는 모델 출력입니다.

In [ ]:
video = "b02_sn_h_02"
frame = 35
figure = plot_pair(pair, video, frame)
figure.savefig(output / f"{video}_frame{frame}_comparison.png", dpi=150)
display(figure)
import matplotlib.pyplot as plt
plt.close(figure)

## FiftyOne에서 프레임 고르기
`open_view`를 `True`로 바꿔 실행합니다. `clarity`, `person_id`, `relative_box_area`로 조건을 고르거나 `change`에서 손실을 선택하세요. `size_640`과 `size_800`은 시간 연결을 적용한 최종 선택입니다. 원본 프레임은 로컬 실험 폴더에 저장됩니다.

In [ ]:
open_view = False
if open_view:
    import fiftyone as fo
    dataset = fiftyone_dataset(pair, output)
    session = fo.launch_app(dataset)

## b02 32~35번 · 박스 크기가 줄어든 위치

노란 점선은 수동 바깥 박스, 분홍선은 원본 밝기로 근사한 안쪽 면, 파란 선은 저장된 선택 박스입니다. 세 줄을 같은 배율로 봅니다. IoU는 각각의 기준을 감싸는 사각형으로 계산합니다. 밝기 130을 표와 그림에 쓰고 110·150에서도 순서가 바뀌는지 확인합니다. 35번에서는 밝은 축 일부가 연결되므로 안쪽 면은 근삿값입니다.

In [ ]:
from wlcv.evaluation.resolution_edges import compare_bright_edges

edges, edge_figure = compare_bright_edges(pair)
edges.to_csv(output / "edge_comparison.csv", index=False)
edge_figure.savefig(output / "edge_comparison.png", dpi=125)
display(edge_figure)
plt.close(edge_figure)

columns = ["frame_index", "imgsz", "source", "pred_width", "pred_height", "iou_inner", "iou_outer"]
display(edges.loc[edges.cutoff.eq(130), columns].round(3))
# 양수면 밝은 면의 박스와 더 많이 겹친다. 세 밝기 기준에서 부호를 비교한다.
difference = edges.assign(inner_minus_outer=edges.iou_inner - edges.iou_outer)
display(difference.groupby(["frame_index", "imgsz"]).inner_minus_outer.agg(["min", "max"]).round(3))